In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 80)

In [ ]:
def load_profile(path):
    """Load a trtexec --exportProfile JSON into a DataFrame.
    Element [0] is a header {"count": N} (iteration count) — skip it.
    Every real entry has: name, timeMs, averageMs, medianMs, percentage."""
    with open(path) as f:
        raw = json.load(f)

    # header is the element WITHOUT a 'name' key ({"count": N})
    count = next((e["count"] for e in raw if "count" in e), None)
    entries = [e for e in raw if "name" in e]        # drops the {"count":…} header

    df = pd.DataFrame(entries)
    # enforce column order / dtypes
    df = df[["name", "timeMs", "averageMs", "medianMs", "percentage"]]
    for col in ["timeMs", "averageMs", "medianMs", "percentage"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    print(f"{Path(path).name}: {len(df)} kernels  (profiled iterations = {count})")
    return df

In [ ]:
import os
for root, dirs, files in os.walk("/workspace"):
    for f in files:
        if f.endswith(".json") and "per_kernel" in f.lower():
            print(os.path.join(root, f))

In [ ]:
QAT_PATH  = "/workspace/Per_kernel_json_file/qat_batch32_per_kernel_profile.json"
PTQ_PATH  = "/workspace/Per_kernel_json_file/ptq_int8_per_kernel_profile.json"
FP16_PATH = "/workspace/Per_kernel_json_file/fp16_per_kernel_profile.json"

df_qat  = load_profile(QAT_PATH)    # expect 245 kernels
df_ptq  = load_profile(PTQ_PATH)    # expect 189 kernels
df_fp16 = load_profile(FP16_PATH)   # expect 231 kernels  (same-arch, NO Q/DQ baseline)
df_qat.head(10)

In [ ]:
def common_columns(df_a, df_b):
    common = sorted(set(df_a.columns) & set(df_b.columns))
    only_a = sorted(set(df_a.columns) - set(df_b.columns))
    only_b = sorted(set(df_b.columns) - set(df_a.columns))
    print(f"Common columns ({len(common)}): {common}")
    if only_a: print(f"Only in QAT: {only_a}")
    if only_b: print(f"Only in PTQ: {only_b}")
    return common

common_cols = common_columns(df_qat, df_ptq)   # should be all 5: name, timeMs, averageMs, medianMs, percentage

In [ ]:
df_qat_c  = df_qat[common_cols].copy();  df_qat_c["engine"]  = "QAT"
df_ptq_c  = df_ptq[common_cols].copy();  df_ptq_c["engine"]  = "PTQ"
df_fp16_c = df_fp16[common_cols].copy(); df_fp16_c["engine"] = "FP16"

print("QAT:", df_qat_c.shape, "| PTQ:", df_ptq_c.shape, "| FP16:", df_fp16_c.shape)

In [ ]:
def summarize(df, label):
    print(f"\n=== {label}  ({len(df)} kernels) ===")
    print(f"  sum(averageMs)   = {df['averageMs'].sum():.4f} ms   <- per-inference total")
    print(f"  sum(medianMs)    = {df['medianMs'].sum():.4f} ms")
    print(f"  sum(percentage)  = {df['percentage'].sum():.2f} %")
    print(f"  top-5 kernels by averageMs:")
    top = df.nlargest(5, "averageMs")[["name", "averageMs", "percentage"]]
    for _, r in top.iterrows():
        print(f"    {r['averageMs']:.4f} ms ({r['percentage']:.1f}%)  {r['name'][:70]}")

summarize(df_qat_c, "QAT")
summarize(df_ptq_c, "PTQ")

In [ ]:
def op_category(name):
    """Bucket each kernel by operation type, from its name."""
    n = name.lower()
    if "pwn(sigmoid" in n or "silu" in n:
        if "conv" in n:
            return "conv+SiLU (fused)"
        return "SiLU standalone"
    if "conv" in n:
        return "conv only"
    if "reformat" in n or "shuffle" in n:
        return "reformat"
    if "topk" in n or "nms" in n or "gather" in n:
        return "NMS/TopK"
    if "softmax" in n or "attn" in n or "matmul" in n:
        return "attention"
    return "other"

for df in (df_qat_c, df_ptq_c):
    df["op_category"] = df["name"].apply(op_category)

# op-category breakdown per engine (this is the conv+SiLU fusion story)
op_qat = df_qat_c.groupby("op_category").agg(ms=("averageMs","sum"), n=("averageMs","size"))
op_ptq = df_ptq_c.groupby("op_category").agg(ms=("averageMs","sum"), n=("averageMs","size"))
op_compare = op_qat.join(op_ptq, lsuffix="_QAT", rsuffix="_PTQ", how="outer").fillna(0)
op_compare["Δms"] = op_compare["ms_QAT"] - op_compare["ms_PTQ"]
op_compare.round(4)

In [ ]:
import re

def layer_key(name):
    """Extract a stable layer identity that can be matched across QAT and PTQ,
    stripping the Q/DQ quantizer suffixes that only QAT has.
    e.g. 'model.2.cv1.conv.weight + /model.2/cv1/conv/weight_quantizer/QuantizeLinear + ...'
         -> 'model.2.cv1.conv'
    """
    n = name
    # strip everything after the first ' + ' (the Q/DQ chain QAT adds)
    n = n.split(" + ")[0]
    # normalize: pull the model.N...conv / op path
    n = n.replace("/", ".").strip(".")
    # collapse the leading path form to a canonical key
    m = re.search(r"(model\.[\w\.]+?)(\.weight|\.conv|$)", n)
    return m.group(1) if m else n.strip()

# apply to both
for df in (df_qat_c, df_ptq_c):
    df["layer_key"] = df["name"].apply(layer_key)

df_qat_c[["name", "layer_key", "averageMs"]].head(10)

In [ ]:
qat_by_layer = (df_qat_c.groupby("layer_key")["averageMs"]
                .agg(qat_ms="sum", qat_n="size"))
ptq_by_layer = (df_ptq_c.groupby("layer_key")["averageMs"]
                .agg(ptq_ms="sum", ptq_n="size"))
print(f"QAT unique layers: {len(qat_by_layer)}  |  PTQ unique layers: {len(ptq_by_layer)}")

In [ ]:
# inner join = ONLY layers present in BOTH engines
common = qat_by_layer.join(ptq_by_layer, how="inner")
common["Δms"] = common["qat_ms"] - common["ptq_ms"]
common["Δpct"] = (common["Δms"] / common["ptq_ms"] * 100).round(1)   # % extra vs PTQ
common = common.sort_values("Δms", ascending=False)

print(f"Common layers matched in BOTH engines: {len(common)}")
print(f"Total extra latency on common layers: {common['Δms'].sum():.4f} ms")
common.round(5)

In [ ]:
# summary report: per-layer extra as % of the total common-layer time
report = common.copy()
report["qat_share_%"] = (report["qat_ms"] / report["qat_ms"].sum() * 100).round(1)
report["extra_share_%"] = (report["Δms"] / report["Δms"].sum() * 100).round(1)  # where the extra concentrates

print("=== Per-layer latency comparison (common layers only) ===")
print(f"Common-layer QAT total: {report['qat_ms'].sum():.4f} ms")
print(f"Common-layer PTQ total: {report['ptq_ms'].sum():.4f} ms")
print(f"Extra on common layers: {report['Δms'].sum():.4f} ms\n")
print("Top layers where QAT loses the most latency:")
report.head(15).round(4)

In [ ]:
# The QAT-ONLY kernels — where the penalty actually lives
qat_only_keys = set(df_qat_c["layer_key"]) - set(df_ptq_c["layer_key"])
qat_only_df = df_qat_c[df_qat_c["layer_key"].isin(qat_only_keys)]

# categorize them
def kind(name):
    n = name.lower()
    if "pwn" in n and "sigmoid" in n: return "SiLU standalone (fusion evicted)"
    if "mulminmaxroun" in n: return "Q/DQ (quantize/dequant)"
    if "reformat" in n or "repl" in n: return "reformat"
    if "topk" in n: return "NMS/TopK"
    return "other"

qat_only_df = qat_only_df.copy()
qat_only_df["kind"] = qat_only_df["name"].apply(kind)

summary = qat_only_df.groupby("kind").agg(ms=("averageMs","sum"), n=("averageMs","size")).sort_values("ms", ascending=False)
print(f"QAT-ONLY kernels: {len(qat_only_df)} kernels, {qat_only_df['averageMs'].sum():.4f} ms total")
print("(this is where QAT's penalty lives — the kernels PTQ doesn't have)\n")
print(summary.round(4))

In [ ]:
other_df = qat_only_df[qat_only_df["kind"] == "other"]
print(f"'other' = {len(other_df)} kernels, {other_df['averageMs'].sum():.4f} ms\n")
for _, r in other_df.nlargest(25, "averageMs").iterrows():
    print(f"  {r['averageMs']:.5f} ms  {r['name'][:75]}")

In [ ]:
def kind(name):
    n = name.lower()
    if "attn" in n or "matmul" in n or "/qkv/" in n or ".attn." in n:
        return "attention"
    if "pwn" in n and "sigmoid" in n:
        return "SiLU standalone (fusion evicted)"
    if "mulminmaxroun" in n:
        return "Q/DQ (quantize/dequant)"
    if "conv.weight" in n or "/conv/" in n:
        return "conv (quantized)"
    if "reformat" in n or "repl" in n:
        return "reformat"
    if "topk" in n:
        return "NMS/TopK"
    return "other"

qat_only_df["kind"] = qat_only_df["name"].apply(kind)
summary = qat_only_df.groupby("kind").agg(ms=("averageMs","sum"), n=("averageMs","size")).sort_values("ms", ascending=False)
print(summary.round(4))

In [ ]:
# take a QAT-only conv and search for its equivalent in PTQ
sample = "model.16.m.0.m.0.cv1"   # one of the QAT-only convs
print("QAT kernels matching:")
print(df_qat_c[df_qat_c["name"].str.contains("16.m.0.m.0.cv1", regex=False)]["name"].tolist())
print("\nPTQ kernels matching (same layer?):")
print(df_ptq_c[df_ptq_c["name"].str.contains("16/m.0/m/m.0/cv1", regex=False)]["name"].tolist())

In [ ]:
# group ALL kernels by their layer, regardless of how many kernels or fusion pattern
def layer_group(name):
    m = re.search(r"model[./](\d+)([./][\w./]*?)(?:[./](?:cv\d|conv|act|attn|m)[\w./]*)?", name)
    if m:
        return f"model.{m.group(1)}" + (("." + m.group(2).replace("/",".").strip(".")) if m.group(2) else "")
    return "non_model"

# simplest robust version: just the model.N.cvX level
def layer_group(name):
    m = re.search(r"model[./](\d+(?:[./]\w+)*?[./]cv\d)", name)
    if m:
        return m.group(1).replace("/", ".")
    m2 = re.search(r"model[./](\d+)", name)
    return f"model.{m2.group(1)}" if m2 else "non_model"

for df in (df_qat_c, df_ptq_c):
    df["layer_group"] = df["name"].apply(layer_group)

qg = df_qat_c.groupby("layer_group")["averageMs"].sum()
pg = df_ptq_c.groupby("layer_group")["averageMs"].sum()
cmp = pd.DataFrame({"qat_ms": qg, "ptq_ms": pg}).fillna(0)
cmp["Δms"] = cmp["qat_ms"] - cmp["ptq_ms"]
cmp.sort_values("Δms", ascending=False).round(5)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 10
COL_QAT  = "#c1272d"   # red
COL_PTQ  = "#0070c0"   # blue
COL_FP16 = "#2ca02c"   # green (same-arch, no-Q/DQ baseline)

# make sure kind is applied (from your refined categorizer)
def kind(name):
    n = name.lower()
    if "attn" in n or "matmul" in n or "/qkv/" in n: return "attention"
    if "pwn" in n and "sigmoid" in n: return "SiLU standalone"
    if "mulminmaxroun" in n: return "Q/DQ"
    if "conv.weight" in n or "/conv/" in n: return "conv"
    if "reformat" in n or "repl" in n: return "reformat"
    if "topk" in n: return "NMS/TopK"
    return "other"

for df in (df_qat_c, df_ptq_c, df_fp16_c):
    df["kind"] = df["name"].apply(kind)

In [ ]:
cat_qat  = df_qat_c.groupby("kind")["averageMs"].sum()
cat_ptq  = df_ptq_c.groupby("kind")["averageMs"].sum()
cat_fp16 = df_fp16_c.groupby("kind")["averageMs"].sum()
cats = sorted(set(cat_qat.index) | set(cat_ptq.index) | set(cat_fp16.index))
qat_vals  = [cat_qat.get(c, 0)  for c in cats]
ptq_vals  = [cat_ptq.get(c, 0)  for c in cats]
fp16_vals = [cat_fp16.get(c, 0) for c in cats]

x = np.arange(len(cats)); w = 0.27
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w, qat_vals,  w, label="QAT",  color=COL_QAT)
ax.bar(x,     ptq_vals,  w, label="PTQ",  color=COL_PTQ)
ax.bar(x + w, fp16_vals, w, label="FP16", color=COL_FP16)
ax.set_xticks(x); ax.set_xticklabels(cats, rotation=30, ha="right")
ax.set_ylabel("Sum averageMs (per-inference)")
ax.set_title("Latency by operation category — QAT vs PTQ vs FP16")
ax.legend()
plt.tight_layout(); plt.savefig("chart_op_category.png", bbox_inches="tight"); plt.show()

In [ ]:
qat_only_keys = set(df_qat_c["layer_key"]) - set(df_ptq_c["layer_key"])
qo = df_qat_c[df_qat_c["layer_key"].isin(qat_only_keys)].copy()
qo["kind"] = qo["name"].apply(kind)
pen = qo.groupby("kind")["averageMs"].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 7))
colors = plt.cm.Reds(np.linspace(0.4, 0.9, len(pen)))
wedges, texts, autotexts = ax.pie(
    pen.values, labels=pen.index, autopct=lambda p: f"{p:.0f}%\n({p/100*pen.sum():.3f}ms)",
    colors=colors, startangle=90, textprops={"fontsize": 9})
ax.set_title(f"QAT's extra-kernel penalty by cause\n(total {pen.sum():.3f} ms across {len(qo)} QAT-only kernels)")
plt.tight_layout(); plt.savefig("chart_penalty_pie.png", bbox_inches="tight"); plt.show()

In [ ]:
n_qat  = df_qat_c.groupby("kind").size()
n_ptq  = df_ptq_c.groupby("kind").size()
n_fp16 = df_fp16_c.groupby("kind").size()
cats = sorted(set(n_qat.index) | set(n_ptq.index) | set(n_fp16.index))
qn = [n_qat.get(c,0) for c in cats]
pn = [n_ptq.get(c,0) for c in cats]
fn = [n_fp16.get(c,0) for c in cats]

x = np.arange(len(cats)); w = 0.27
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x-w, qn, w, label=f"QAT ({sum(qn)} kernels)",  color=COL_QAT)
ax.bar(x,   pn, w, label=f"PTQ ({sum(pn)} kernels)",  color=COL_PTQ)
ax.bar(x+w, fn, w, label=f"FP16 ({sum(fn)} kernels)", color=COL_FP16)
ax.set_xticks(x); ax.set_xticklabels(cats, rotation=30, ha="right")
ax.set_ylabel("Number of kernels")
ax.set_title("Kernel COUNT by category — QAT has more jobs (launch-bound)")
ax.legend()
plt.tight_layout(); plt.savefig("chart_kernel_counts.png", bbox_inches="tight"); plt.show()

In [ ]:
import re
def lg(name):
    m = re.search(r"model[./](\d+)", name)
    if m: return f"model.{m.group(1)}"
    n = name.lower()
    if "pwn" in n and "sigmoid" in n: return "SiLU_extra"
    if "mulminmaxroun" in n: return "Q/DQ_extra"
    return "other_extra"

for df in (df_qat_c, df_ptq_c):
    df["lg"] = df["name"].apply(lg)
qg = df_qat_c.groupby("lg")["averageMs"].sum()
pg = df_ptq_c.groupby("lg")["averageMs"].sum()
d = pd.DataFrame({"qat": qg, "ptq": pg}).fillna(0)
d["delta"] = d["qat"] - d["ptq"]
d = d.sort_values("delta")

fig, ax = plt.subplots(figsize=(9, 10))
colors = [COL_QAT if v > 0 else COL_PTQ for v in d["delta"]]
ax.barh(d.index, d["delta"], color=colors)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Δms (QAT − PTQ)   ← QAT faster | QAT slower →")
ax.set_title("Per-layer-group latency difference\n(red = QAT slower, blue = QAT faster)")
plt.tight_layout(); plt.savefig("chart_per_layer_delta.png", bbox_inches="tight"); plt.show()

In [ ]:
import os
print("Saved charts:")
for f in ["chart_op_category.png","chart_penalty_pie.png","chart_kernel_counts.png","chart_per_layer_delta.png"]:
    print(f"  {f}  ({os.path.getsize(f)//1024} KB)")